# Final Model Loading and Saving Check

In this notebook, I will load the saved final machine learning pipeline and verify that it works correctly.

The saved pipeline includes both preprocessing steps and the tuned Random Forest model.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

data_path = "../data/brfss_diabetes_cleaned.csv"
model_path = "../models/diabetes_rf_pipeline.pkl"

print("Data file exists:", os.path.exists(data_path))
print("Model file exists:", os.path.exists(model_path))

Data file exists: True
Model file exists: True


## Load Cleaned Data and Saved Model

In this step, I will load the cleaned dataset and the saved final Random Forest pipeline.

In [2]:
df = pd.read_csv(data_path)
final_model = joblib.load(model_path)

print("Data loaded successfully!")
print("Data shape:", df.shape)

print("\nModel loaded successfully!")
print(type(final_model))

Data loaded successfully!
Data shape: (453241, 14)

Model loaded successfully!
<class 'sklearn.pipeline.Pipeline'>


In [3]:
final_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['BMI', 'GeneralHealth',
                                                   'PhysicalHealthDays',
                                                   'MentalHealthDays', 'Stroke',
                                                   'HeartDisease', 'Smoker',
                                                   'PhysicalActivity', 'Sex',
                                                   'Age', 'Education', 'Income',
                                                   'HeavyAlcohol'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced', max_depth=12,
                                        min_samples_leaf=4, min_samples_split=5,
                                        n_estimators=150, n_jobs=-1,
                                        random_state=42))])

## Test Saved Model Predictions

In this step, I will test the saved pipeline by making predictions on a small sample from the dataset.

In [4]:
X = df.drop("Diabetes_012", axis=1)
y = df["Diabetes_012"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (453241, 13)
y shape: (453241,)


In [5]:
sample_data = X.head(5)
sample_actual = y.head(5)

sample_predictions = final_model.predict(sample_data)
sample_probabilities = final_model.predict_proba(sample_data)[:, 1]

prediction_check = sample_data.copy()
prediction_check["Actual"] = sample_actual.values
prediction_check["Predicted"] = sample_predictions
prediction_check["Diabetes_Probability"] = sample_probabilities

prediction_check

,BMI,GeneralHealth,PhysicalHealthDays,MentalHealthDays,Stroke,HeartDisease,Smoker,PhysicalActivity,Sex,Age,Education,Income,HeavyAlcohol,Actual,Predicted,Diabetes_Probability
0,22.49,1.0,2.0,0.0,0.0,0.0,0.0,1.0,0.0,12.0,2.0,NaN,0.0,0.0,0.0,0.342014
1,25.83,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,13.0,4.0,7.0,0.0,0.0,1.0,0.547491
2,22.53,1.0,30.0,0.0,0.0,0.0,1.0,1.0,1.0,8.0,3.0,NaN,0.0,0.0,0.0,0.412113
3,25.09,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,13.0,4.0,4.0,0.0,0.0,0.0,0.440114
4,19.77,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,6.0,3.0,2.0,0.0,0.0,0.0,0.248291


## Evaluate Loaded Final Model on Test Set

In this step, I will evaluate the loaded final model on the test set to confirm that the saved pipeline gives the same expected performance.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (90649, 13)
y_test shape: (90649,)


In [7]:
y_pred_loaded = final_model.predict(X_test)
y_pred_proba_loaded = final_model.predict_proba(X_test)[:, 1]

print("Loaded model predictions completed!")
print("Predictions shape:", y_pred_loaded.shape)
print("Probabilities shape:", y_pred_proba_loaded.shape)

Loaded model predictions completed!
Predictions shape: (90649,)
Probabilities shape: (90649,)


In [8]:
loaded_accuracy = accuracy_score(y_test, y_pred_loaded)
loaded_precision = precision_score(y_test, y_pred_loaded)
loaded_recall = recall_score(y_test, y_pred_loaded)
loaded_f1 = f1_score(y_test, y_pred_loaded)
loaded_roc_auc = roc_auc_score(y_test, y_pred_proba_loaded)

print("Loaded Final Model Evaluation:")
print("Accuracy:", round(loaded_accuracy, 4))
print("Precision:", round(loaded_precision, 4))
print("Recall:", round(loaded_recall, 4))
print("F1-score:", round(loaded_f1, 4))
print("ROC-AUC:", round(loaded_roc_auc, 4))

Loaded Final Model Evaluation:
Accuracy: 0.7004
Precision: 0.2891
Recall: 0.7291
F1-score: 0.414
ROC-AUC: 0.7866


## Save Final Model Metrics

In this step, I will save the final model performance metrics into a CSV file for documentation and deployment use.

In [9]:
final_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Score": [
        loaded_accuracy,
        loaded_precision,
        loaded_recall,
        loaded_f1,
        loaded_roc_auc
    ]
})

final_metrics["Score"] = final_metrics["Score"].round(4)

metrics_path = "../models/final_model_metrics.csv"

final_metrics.to_csv(metrics_path, index=False)

print("Final model metrics saved successfully!")
print("Metrics path:", metrics_path)

final_metrics

Final model metrics saved successfully!
Metrics path: ../models/final_model_metrics.csv


,Metric,Score
0,Accuracy,0.7004
1,Precision,0.2891
2,Recall,0.7291
3,F1-score,0.4140
4,ROC-AUC,0.7866


In [10]:
print(os.path.exists("../models/final_model_metrics.csv"))

True
